# Mini Project Database — Aplikasi Kantin Terintegrasi (Kelompok 11)

Notebook ini berisi implementasi aplikasi **Kantin Terintegrasi** berbasis
SQLite dan Gradio. Aplikasi ini memiliki dua peran pengguna, yaitu
**Pembeli** dan **Penjual**, yang masing-masing memiliki hak akses dan
fitur yang berbeda terhadap basis data.

Secara garis besar, notebook ini disusun ke dalam beberapa bagian utama:

1. **Persiapan lingkungan** — instalasi/pemanggilan pustaka dan pengunduhan
   basis data dari Google Drive.
2. **Konfigurasi global** — variabel dan konstanta yang digunakan di
   seluruh aplikasi (path basis data, keranjang belanja, kategori menu).
3. **Modul autentikasi** — fungsi login dan registrasi akun.
4. **Modul halaman pembeli** — fungsi untuk melihat dan memperbarui profil,
   melihat daftar kios/menu, mengelola keranjang, dan melakukan checkout.
5. **Modul halaman penjual** — fungsi untuk melihat profil kios, mengelola
   menu (tambah/ubah/hapus), dan melihat riwayat transaksi.
6. **Antarmuka pengguna (UI) berbasis Gradio** — perakitan seluruh fungsi
   di atas ke dalam tampilan aplikasi web interaktif.

Setiap bagian kode dilengkapi dengan sel markdown yang menjelaskan tujuan
sel tersebut serta interpretasi/implikasi dari output yang dihasilkan,
sehingga alur logika aplikasi dapat lebih mudah dipahami oleh pembaca.


## 1. Persiapan Lingkungan

### 1.1 Import Pustaka

Sel di bawah ini mengimpor seluruh pustaka yang dibutuhkan sepanjang
notebook:

- `gdown` — mengunduh berkas basis data (`database.db`) yang tersimpan di
  Google Drive.
- `sqlite3` — pustaka bawaan Python untuk berinteraksi dengan basis data
  SQLite (menjalankan query `SELECT`, `INSERT`, `UPDATE`, dan `DELETE`).
- `gradio` — membangun antarmuka pengguna (UI) berbasis web secara cepat
  tanpa perlu menulis kode HTML/CSS/JavaScript secara manual.
- `datetime` — menghasilkan tanggal transaksi yang digunakan sebagai bagian
  dari ID transaksi pada proses checkout.

**Implikasi:** apabila salah satu pustaka (khususnya `gdown` dan `gradio`)
belum tersedia pada environment, sel ini akan menghasilkan `ModuleNotFoundError`.
Pada Google Colab, kedua pustaka umumnya perlu diinstal terlebih dahulu
melalui `!pip install gdown gradio`.


In [ ]:
import gdown
import sqlite3
import gradio as gr
from datetime import datetime


### 1.2 Pengunduhan Basis Data dari Google Drive

Sel berikut mendefinisikan dua fungsi bantu dan langsung menjalankannya
untuk mengunduh basis data:

- `get_direct_gdrive_link(share_url)` — mengonversi tautan berbagi
  (*share link*) Google Drive menjadi tautan unduh langsung
  (`https://drive.google.com/uc?id=<file_id>`), dengan mengekstrak
  `file_id` dari URL yang diberikan.
- `download_database(url, output='database.db')` — memanggil `gdown` untuk
  mengunduh berkas dari tautan langsung tersebut dan menyimpannya secara
  lokal dengan nama `database.db`.

Fungsi ini kemudian dipanggil langsung terhadap `google_drive_url` yang
telah ditentukan, sehingga basis data akan tersedia secara lokal di
runtime (`db_file`) sebelum digunakan oleh fungsi-fungsi lain di notebook
ini.

**Implikasi/interpretasi output:** keluaran sel ini berupa log progres
unduhan dari `gdown`. Jika proses berhasil, berkas `database.db` akan
muncul di direktori kerja (`/content/` pada Google Colab) dan siap
diakses oleh `sqlite3`. Jika tautan Google Drive tidak lagi bersifat
publik/dapat diakses, sel ini akan gagal dan proses selanjutnya
(yang bergantung pada `DB_PATH`) tidak akan berjalan.


In [ ]:
# Tautan berbagi (share link) basis data pada Google Drive
google_drive_url = 'https://drive.google.com/file/d/1fbJVLnf51FIav0zj3tYE8p7nagkCbofA/view?usp=drive_link'


def get_direct_gdrive_link(share_url):
    """Mengonversi tautan berbagi Google Drive menjadi tautan unduh langsung."""
    file_id = share_url.split('/d/')[1].split('/')[0]
    return f'https://drive.google.com/uc?id={file_id}'


def download_database(url, output='database.db'):
    """Mengunduh basis data dari Google Drive dan menyimpannya secara lokal."""
    direct_link = get_direct_gdrive_link(url)
    gdown.download(direct_link, output, quiet=False)
    return output


# Mengunduh basis data agar dapat diakses oleh seluruh fungsi pada notebook ini
db_file = download_database(google_drive_url)


## 2. Konfigurasi Global

Sel berikut mendefinisikan variabel dan konstanta global yang dipakai di
seluruh bagian aplikasi:

- `DB_PATH` — lokasi berkas basis data SQLite yang telah diunduh
  sebelumnya. Seluruh fungsi yang mengakses basis data akan merujuk pada
  path ini.
- `cart` — list kosong yang berperan sebagai keranjang belanja sementara
  (state) milik pembeli selama sesi aplikasi berjalan.
- `payment_method` — variabel string yang menyimpan metode pembayaran yang
  dipilih pembeli sebelum melakukan konfirmasi order.
- `ALLOWED_CATEGORIES` — daftar kategori menu yang diperbolehkan
  (*whitelist*), digunakan untuk memvalidasi input kategori pada saat
  penjual menambah atau mengubah menu.

**Implikasi:** karena `cart` dan `payment_method` merupakan variabel global
yang dimodifikasi lintas fungsi (menggunakan `global payment_method` pada
fungsi terkait), state keranjang belanja bersifat tunggal untuk seluruh
sesi aplikasi (tidak dipisahkan per pengguna). Hal ini merupakan
penyederhanaan yang wajar untuk lingkup mini project, namun perlu menjadi
catatan apabila aplikasi diakses oleh lebih dari satu pengguna secara
bersamaan.


In [ ]:
DB_PATH = '/content/database.db'

# State sementara (in-memory) untuk keranjang belanja dan metode pembayaran
cart = []
payment_method = ""

# Kategori menu yang diperbolehkan pada saat penambahan/pengeditan menu
ALLOWED_CATEGORIES = ["Makanan berat", "Makanan ringan", "Minuman"]


## 3. Modul Autentikasi

### 3.1 Fungsi `cek_login`

Fungsi `cek_login(email, password)` memverifikasi kredensial pengguna
terhadap tabel `Akun`, kemudian menentukan peran pengguna (`penjual` atau
`pembeli`) dengan mengecek keberadaan `ID Akun` pada tabel `Kios` maupun
tabel `Pembeli`.

Alur logikanya:
1. Mencari `ID Akun` pada tabel `Akun` berdasarkan `Email` dan `Password`
   yang dimasukkan. Jika tidak ditemukan, fungsi mengembalikan `(None, None)`.
2. Jika `ID Akun` ditemukan pada tabel `Kios`, peran pengguna adalah
   `"penjual"`.
3. Jika ditemukan pada tabel `Pembeli`, peran pengguna adalah `"pembeli"`.
4. Jika `ID Akun` ada di tabel `Akun` namun tidak terhubung ke `Kios`
   maupun `Pembeli`, statusnya `"tidak_diketahui"` (kondisi data tidak
   konsisten).

**Implikasi:** fungsi ini menjadi gerbang akses utama aplikasi. Nilai
kembalian `(id_akun, role)` digunakan untuk menentukan tampilan (halaman
pembeli atau penjual) yang akan ditampilkan pada UI Gradio setelah login
berhasil.


In [ ]:
# --- AUTENTIKASI ---

def cek_login(email, password):
    """Memverifikasi kredensial akun dan menentukan peran pengguna
    (penjual/pembeli) berdasarkan keterhubungannya dengan tabel Kios/Pembeli.
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Akun" FROM Akun WHERE "Email"=? AND "Password"=?', (email, password))
    user = cursor.fetchone()

    if not user:
        conn.close()
        return None, None

    id_akun = user[0]

    cursor.execute('SELECT 1 FROM Kios WHERE "ID Akun"=?', (id_akun,))
    if cursor.fetchone():
        conn.close()
        return id_akun, "penjual"

    cursor.execute('SELECT 1 FROM Pembeli WHERE "ID Akun"=?', (id_akun,))
    if cursor.fetchone():
        conn.close()
        return id_akun, "pembeli"

    conn.close()
    return id_akun, "tidak_diketahui"


### 3.2 Fungsi `register_akun`

Fungsi `register_akun` menangani proses pendaftaran akun baru, baik
sebagai `pembeli` maupun `penjual`. Fungsi ini melakukan beberapa lapis
validasi sebelum menyisipkan data ke basis data:

1. **Validasi kelengkapan input** — memastikan `email`, `password`,
   `role`, `nama`, dan `no_hp_or_pedagang` tidak kosong.
2. **Validasi khusus peran** — untuk `pembeli`, `jurusan_or_kios` (Jurusan)
   dan `npm` wajib diisi; untuk `penjual`, `jurusan_or_kios` (Nama Kios)
   wajib diisi.
3. **Validasi duplikasi email** — menolak registrasi apabila email sudah
   terdaftar pada tabel `Akun`.
4. **Penyisipan data** — menyisipkan baris baru ke tabel `Akun`, kemudian
   ke tabel `Pembeli` atau `Kios` sesuai peran yang dipilih. Khusus
   `penjual`, `ID Kios` baru dihasilkan secara manual dengan mengambil
   nilai maksimum `ID Kios` yang ada lalu menambah 1 (dimulai dari 100
   apabila tabel `Kios` masih kosong).

Seluruh proses dibungkus dalam blok `try-except` sehingga kegagalan pada
level basis data (misalnya constraint violation) tidak menghentikan
aplikasi, melainkan dikembalikan sebagai pesan error yang informatif.

**Implikasi:** skema penomoran `ID Kios` (mulai dari 100, increment 1)
menjadi dasar bagi skema penomoran `ID Menu` pada modul penjual
(lihat bagian `tambah_menu`), sehingga terdapat keterkaitan langsung antara
`ID Kios` dan rentang `ID Menu` yang dimiliki sebuah kios.


In [ ]:
def register_akun(email, password, role, nama, jurusan_or_kios, no_hp_or_pedagang, npm):
    """Mendaftarkan akun baru sebagai pembeli atau penjual, lengkap dengan
    validasi input dan validasi duplikasi email.
    """
    try:
        # Validasi input kosong
        if not all([email, password, role, nama, no_hp_or_pedagang]):
            return "Email, Password, Role, Nama, dan No HP/Nama Pedagang harus diisi untuk melanjutkan registrasi."

        if role == "pembeli" and (not jurusan_or_kios or not npm):
            return "Jurusan dan NPM harus diisi untuk pendaftar sebagai pembeli."

        if role == "penjual" and not jurusan_or_kios:
            return "Nama Kios harus diisi untuk pendaftar sebagai penjual."

        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()

        # Cek apakah email sudah terdaftar
        cursor.execute('SELECT 1 FROM Akun WHERE Email = ?', (email,))
        if cursor.fetchone() is not None:
            conn.close()
            return "Email sudah terdaftar. Gunakan email lain atau login."

        # Insert ke tabel Akun
        cursor.execute('INSERT INTO Akun (Email, Password) VALUES (?, ?)', (email, password))
        id_akun = cursor.lastrowid

        if role == "pembeli":
            cursor.execute(
                'INSERT INTO Pembeli (NPM, Nama, Jurusan, "Nomor Telepon", "ID Akun") VALUES (?, ?, ?, ?, ?)',
                (npm, nama, jurusan_or_kios, no_hp_or_pedagang, id_akun)
            )
        elif role == "penjual":
            cursor.execute('SELECT MAX("ID Kios") FROM Kios')
            last_id_kios = cursor.fetchone()[0]
            new_id_kios = 100 if last_id_kios is None else last_id_kios + 1
            cursor.execute(
                'INSERT INTO Kios ("ID Kios", "Nama Kios", "Nama Pedagang", "ID Akun") VALUES (?, ?, ?, ?)',
                (new_id_kios, jurusan_or_kios, no_hp_or_pedagang, id_akun)
            )
        else:
            conn.close()
            return "Role tidak valid."

        conn.commit()
        conn.close()
        return "Registrasi berhasil. Silakan login."
    except Exception as e:
        return f"Error saat registrasi: {e}"


## 4. Modul Halaman Pembeli

### 4.1 Fungsi Profil Pembeli

Dua fungsi berikut menangani pengelolaan profil pembeli:

- `ambil_profil_pembeli(id_akun)` — mengambil data `NPM`, `Nama`,
  `Jurusan`, dan `Nomor Telepon` dari tabel `Pembeli` berdasarkan
  `ID Akun`, lalu memformatnya menjadi teks yang siap ditampilkan pada UI.
- `ubah_profil_pembeli(id_akun, email_baru, password_baru, telp_baru)` —
  memperbarui data akun (email/password pada tabel `Akun`) dan/atau nomor
  telepon (pada tabel `Pembeli`) secara **parsial**: hanya kolom yang
  diisi (tidak kosong) yang akan diperbarui.

**Implikasi:** karena pembaruan bersifat parsial, pengguna dapat mengubah
salah satu field saja (misalnya hanya nomor telepon) tanpa perlu mengisi
ulang seluruh data. Namun perlu dicatat bahwa fungsi ini tidak melakukan
validasi format (misalnya format email atau nomor telepon), sehingga
validasi format sepenuhnya bergantung pada input pengguna melalui UI.


In [ ]:
# --- HALAMAN PEMBELI ---

def ambil_profil_pembeli(id_akun):
    """Mengambil dan memformat data profil pembeli berdasarkan ID Akun."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT NPM, Nama, Jurusan, "Nomor Telepon" FROM Pembeli WHERE "ID Akun"=?', (int(id_akun),))
    data = cursor.fetchone()
    conn.close()

    if not data:
        return "Data profil tidak ditemukan."

    npm, nama, jurusan, telp = data
    return f"NPM: {npm}\nNama: {nama}\nJurusan: {jurusan}\nNomor Telepon: {telp}"


def ubah_profil_pembeli(id_akun, email_baru, password_baru, telp_baru):
    """Memperbarui data akun dan/atau nomor telepon pembeli secara parsial
    (hanya field yang diisi yang akan diperbarui).
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    try:
        if email_baru:
            cursor.execute('UPDATE Akun SET Email=? WHERE "ID Akun"=?', (email_baru, int(id_akun)))
        if password_baru:
            cursor.execute('UPDATE Akun SET Password=? WHERE "ID Akun"=?', (password_baru, int(id_akun)))
        if telp_baru:
            cursor.execute('UPDATE Pembeli SET "Nomor Telepon"=? WHERE "ID Akun"=?', (telp_baru, int(id_akun)))

        conn.commit()
        return "Profil berhasil diperbarui."
    except Exception as e:
        return f"Gagal memperbarui profil: {e}"
    finally:
        conn.close()


### 4.2 Fungsi Pengambilan Data Kios dan Menu

Tiga fungsi berikut menyediakan data referensi kios dan menu yang
digunakan untuk mengisi komponen dropdown pada UI pembeli:

- `get_list_kios()` — mengembalikan dictionary `{nama_kios: id_kios}`
  untuk seluruh kios yang terdaftar.
- `get_menu_by_kios(id_kios)` — mengembalikan dictionary
  `{nama_menu: id_menu}` untuk seluruh menu milik satu kios tertentu.
- `get_kiosk_name_by_id(id_kios)` — mengembalikan nama kios berdasarkan
  `ID Kios` (digunakan sebagai fungsi bantu pelengkap).

**Implikasi:** pola pengembalian data dalam bentuk dictionary
`{nama: id}` mempermudah integrasi dengan komponen `gr.Dropdown`, karena
pengguna memilih berdasarkan nama (yang mudah dibaca), sedangkan ID yang
sesuai dapat langsung diambil melalui pencarian pada dictionary tersebut
saat diperlukan oleh query berikutnya.


In [ ]:
# --- MENU DAN KIOS ---

def get_list_kios():
    """Mengambil daftar seluruh kios dalam bentuk {nama_kios: id_kios}."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Kios", "Nama Kios" FROM Kios')
    kios = cursor.fetchall()
    conn.close()
    return {nama: id_kios for id_kios, nama in kios}


def get_menu_by_kios(id_kios):
    """Mengambil daftar menu milik satu kios dalam bentuk {nama_menu: id_menu}."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Menu", "Menu" FROM Menu WHERE "ID Kios"=?', (id_kios,))
    menu = cursor.fetchall()
    conn.close()
    return {nama: id_menu for id_menu, nama in menu}


def get_kiosk_name_by_id(id_kios):
    """Mengambil nama kios berdasarkan ID Kios."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "Nama Kios" FROM Kios WHERE "ID Kios"=?', (id_kios,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else "Kios Tidak Ditemukan"


### 4.3 Fungsi Pengelolaan Keranjang Belanja (Cart)

Kumpulan fungsi berikut mengelola state `cart` dan `payment_method` yang
telah didefinisikan pada bagian konfigurasi global:

- `tambah_ke_cart(nama_menu)` — menambahkan nama menu ke dalam `cart`.
- `lihat_cart()` — menampilkan isi `cart` saat ini sebagai teks
  (satu item per baris), atau pesan "Cart kosong." apabila belum ada item.
- `kosongkan_cart()` — mengosongkan seluruh isi `cart`.
- `simpan_metode(metode)` — menyimpan metode pembayaran yang dipilih ke
  variabel global `payment_method`.

**Implikasi:** karena `cart` menyimpan **nama menu**, bukan `ID Menu`,
proses konfirmasi order (`konfirmasi_order`) perlu melakukan pencarian
ulang `ID Menu` berdasarkan nama pada saat checkout. Pendekatan ini cukup
rentan terhadap ambiguitas apabila terdapat dua kios yang memiliki nama
menu yang identik, karena query pencarian `ID Menu` berdasarkan nama tidak
membatasi pencarian pada kios yang sedang dipilih pembeli.


In [ ]:
# --- CART ---

def tambah_ke_cart(nama_menu):
    """Menambahkan satu item menu ke dalam keranjang belanja."""
    cart.append(nama_menu)
    return f"Ditambahkan ke cart: {nama_menu}"


def lihat_cart():
    """Menampilkan seluruh isi keranjang belanja saat ini."""
    if not cart:
        return "Cart kosong."
    return "\n".join(cart)


def kosongkan_cart():
    """Mengosongkan seluruh isi keranjang belanja."""
    cart.clear()
    return "Cart dikosongkan."


def simpan_metode(metode):
    """Menyimpan metode pembayaran yang dipilih pembeli."""
    global payment_method
    payment_method = metode
    return f"Metode pembayaran: {metode}"


### 4.4 Fungsi `konfirmasi_order` (Checkout)

Fungsi ini merupakan inti dari proses checkout pembeli. Alur logikanya:

1. Memvalidasi bahwa `cart` tidak kosong dan `payment_method` sudah
   dipilih.
2. Mengambil `NPM` pembeli berdasarkan `hidden_id` (ID Akun yang tersimpan
   secara tersembunyi pada UI setelah login).
3. **Membentuk ID Transaksi** dengan format `<bulan><hari><urutan>`,
   misalnya `0726003` untuk transaksi ketiga pada tanggal 26 Juli. Nilai
   urutan diperoleh dengan menghitung jumlah `ID Transaksi` unik pada
   tabel `Transaksi Penjualan` untuk tanggal berjalan, ditambah 1.
4. Untuk setiap item pada `cart`, mencari `ID Menu` yang sesuai lalu
   menyisipkan satu baris baru ke tabel `Transaksi Penjualan` (dengan
   jumlah tetap `1` per baris — artinya setiap kali sebuah menu yang sama
   ditambahkan berulang kali ke cart, akan tercatat sebagai baris
   transaksi terpisah, bukan diagregasi menjadi satu baris dengan jumlah
   > 1).
5. Menyimpan seluruh perubahan (`commit`) dan mengembalikan pesan sukses
   berisi ID Transaksi yang terbentuk.

Pada blok `finally`, koneksi basis data selalu ditutup dan `cart`
dikosongkan, terlepas dari apakah proses checkout berhasil atau gagal.

**Implikasi:** skema pembentukan `ID Transaksi` berbasis tanggal dan
penghitungan `COUNT(DISTINCT ...)` memiliki potensi **race condition**
apabila terdapat dua transaksi yang diproses hampir bersamaan (nilai
`urutan_hari_ini` yang sama dapat dihasilkan untuk dua transaksi berbeda).
Untuk lingkup mini project dengan pengguna terbatas, risiko ini relatif
kecil, namun perlu menjadi catatan apabila aplikasi dikembangkan lebih
lanjut untuk skala produksi.


In [ ]:
def konfirmasi_order(hidden_id):
    """Memproses checkout: memvalidasi cart & metode pembayaran, membentuk
    ID Transaksi baru, lalu menyimpan seluruh item cart ke tabel
    Transaksi Penjualan.
    """
    if not cart:
        return "Cart kosong, tidak bisa konfirmasi."
    if not payment_method:
        return "Silakan pilih metode pembayaran terlebih dahulu."

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    try:
        cursor.execute('SELECT NPM FROM Pembeli WHERE "ID Akun"=?', (int(hidden_id),))
        row = cursor.fetchone()
        if not row:
            return "Gagal ambil NPM pembeli."
        npm = row[0]

        # Membentuk ID Transaksi dengan format <bulan><hari><urutan ke-3 digit>
        today_str = datetime.today().strftime('%Y-%m-%d')
        hari = today_str[-2:]
        bulan = today_str[5:7]
        kode_tanggal = f"{bulan}{hari}"

        cursor.execute("SELECT COUNT(DISTINCT 'ID Transaksi') FROM 'Transaksi Penjualan' WHERE Tanggal = ?", (today_str,))
        urutan_hari_ini = cursor.fetchone()[0] + 1
        id_transaksi = f"{kode_tanggal}{str(urutan_hari_ini).zfill(3)}"

        # Menyisipkan setiap item pada cart sebagai satu baris transaksi
        for nama_menu in cart:
            cursor.execute('SELECT "ID Menu" FROM Menu WHERE "Menu"=?', (nama_menu,))
            row = cursor.fetchone()
            if not row:
                continue
            id_menu = row[0]

            cursor.execute("""
                INSERT INTO 'Transaksi Penjualan' (
                    "ID Transaksi", "Tanggal", "NPM", "ID Menu", "Jumlah", "Metode Pembayaran"
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (id_transaksi, today_str, npm, id_menu, 1, payment_method))

        conn.commit()
        return f"🎉 Order berhasil dicatat dengan ID Transaksi: {id_transaksi}"
    except Exception as e:
        return f"❌ Error: {e}"
    finally:
        conn.close()
        cart.clear()


## 5. Modul Halaman Penjual

### 5.1 Fungsi Bantu dan Profil Kios

- `get_id_kios_from_akun(id_akun)` — fungsi bantu yang dipakai hampir di
  seluruh fitur penjual untuk menerjemahkan `ID Akun` yang sedang login
  menjadi `ID Kios` miliknya.
- `lihat_profil_kios(id_akun)` — menampilkan data profil kios (`ID Kios`,
  `Nama Kios`, `Nama Pedagang`, `ID Akun`) dalam format teks.

**Implikasi:** karena hampir seluruh operasi tulis pada modul penjual
(tambah/ubah/hapus menu) terlebih dahulu memanggil
`get_id_kios_from_akun`, fungsi ini secara efektif berperan sebagai
mekanisme otorisasi sederhana — memastikan seorang penjual hanya dapat
mengelola data milik kiosnya sendiri.


In [ ]:
# --- HALAMAN PENJUAL ---

def get_id_kios_from_akun(id_akun):
    """Menerjemahkan ID Akun penjual yang sedang login menjadi ID Kios miliknya."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT "ID Kios" FROM Kios WHERE "ID Akun"=?', (id_akun,))
    result = cursor.fetchone()
    conn.close()
    return result[0] if result else None


def lihat_profil_kios(id_akun):
    """Menampilkan data profil kios milik penjual yang sedang login."""
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        cursor.execute(
            'SELECT "ID Kios", "Nama Kios", "Nama Pedagang", "ID Akun" FROM Kios WHERE "ID Akun"=?',
            (int(id_akun),)
        )
        data = cursor.fetchone()
        conn.close()

        if not data:
            return "Profil kios tidak ditemukan."

        id_kios, nama_kios, nama_pedagang, id_akun = data
        return f"ID Kios: {id_kios}\nNama Kios: {nama_kios}\nNama Pedagang: {nama_pedagang}\nID Akun: {id_akun}"
    except Exception as e:
        return f"Error: {e}"


### 5.2 Fungsi Rekap Data (Menu, Kios, dan Transaksi)

Tiga fungsi berikut menyediakan tampilan rekap data bagi penjual pada tab
"Lihat Data":

- `lihat_semua_menu_penjual(id_akun)` — menampilkan seluruh menu milik
  kios penjual yang bersangkutan, hasil `INNER JOIN` antara tabel `Menu`
  dan `Kios`.
- `lihat_semua_kios()` — menampilkan seluruh data pada tabel `Kios`
  (tidak difilter berdasarkan penjual, sehingga penjual dapat melihat
  kios-kios lain sebagai referensi/pembanding).
- `lihat_trx_penjual(id_akun)` — menampilkan riwayat transaksi yang
  melibatkan menu-menu milik kios penjual tersebut, hasil `INNER JOIN`
  antara tabel `Transaksi Penjualan` dan `Menu`.

**Implikasi:** ketiga fungsi ini bersifat *read-only* dan berperan sebagai
dasbor sederhana bagi penjual untuk memantau kondisi kios, menu, dan
penjualannya. Format keluaran berupa teks mentah (`str(row)` per baris)
cukup memadai untuk kebutuhan mini project, namun kurang ramah dibaca
dibandingkan tabel terstruktur apabila jumlah data cukup besar.


In [ ]:
def lihat_semua_menu_penjual(id_akun):
    """Menampilkan seluruh menu milik kios dari penjual yang sedang login."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    id_kios = get_id_kios_from_akun(id_akun)
    if not id_kios:
        conn.close()
        return "Tidak dapat menemukan ID Kios untuk akun ini."

    query = '''
        SELECT
            Menu."ID Menu",
            Menu."Menu",
            Menu.Kategori,
            Menu.Harga,
            Kios."Nama Kios",
            Kios."Nama Pedagang"
        FROM Menu
        INNER JOIN Kios ON Menu."ID Kios" = Kios."ID Kios"
        WHERE Menu."ID Kios" = ?
    '''
    cursor.execute(query, (id_kios,))
    data = cursor.fetchall()
    conn.close()

    return "\n".join([
        f"{id_menu} - {menu} - {kategori} - Rp{harga} - {nama_kios} (oleh {nama_pedagang})"
        for id_menu, menu, kategori, harga, nama_kios, nama_pedagang in data
    ]) or "Tidak ada menu yang terdaftar untuk kios Anda."


def lihat_semua_kios():
    """Menampilkan seluruh data kios yang terdaftar pada basis data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM Kios')
    data = cursor.fetchall()
    conn.close()
    return "\n".join([str(row) for row in data]) or "Data kosong."


def lihat_trx_penjual(id_akun):
    """Menampilkan riwayat transaksi yang melibatkan menu milik kios penjual."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    id_kios = get_id_kios_from_akun(id_akun)
    if not id_kios:
        conn.close()
        return "Tidak dapat menemukan ID Kios untuk akun ini."

    query = '''
        SELECT
            TP."ID Transaksi",
            TP.Tanggal,
            TP.NPM,
            M.Menu,
            TP.Jumlah,
            TP."Metode Pembayaran"
        FROM "Transaksi Penjualan" AS TP
        INNER JOIN Menu AS M ON TP."ID Menu" = M."ID Menu"
        WHERE M."ID Kios" = ?
    '''
    cursor.execute(query, (id_kios,))
    data = cursor.fetchall()
    conn.close()
    return "\n".join([str(row) for row in data]) or "Tidak ada transaksi untuk kios Anda."


## 6. Antarmuka Pengguna (UI) Berbasis Gradio

Sel berikut merakit seluruh fungsi yang telah didefinisikan sebelumnya ke
dalam satu aplikasi web interaktif menggunakan `gr.Blocks()`. Struktur UI
disusun sebagai berikut:

1. **Halaman Login & Registrasi** — form login (email & password) beserta
   accordion pendaftaran akun baru. Field pendaftaran bersifat dinamis
   (`update_form_fields`): field yang ditampilkan menyesuaikan peran yang
   dipilih (`pembeli` menampilkan NPM & Jurusan, `penjual` menampilkan
   Nama Kios & Nama Pedagang).
2. **Halaman Pembeli** (`pembeli_box`) — berisi dua sub-halaman yang dapat
   dibuka/tutup secara dinamis: **Profil** (lihat & ubah profil) dan
   **Order** (pilih kios & menu, kelola cart, pilih metode pembayaran,
   konfirmasi order).
3. **Halaman Penjual** (`penjual_box`) — berisi beberapa tab: **Lihat
   Data** (menu, kios, transaksi), **Tambah Menu**, **Profil Kios**,
   **Edit Menu**, **Hapus Menu**, dan **Kelola Kios** (memperbarui data
   kios sekaligus akun terkait).
4. **Logika navigasi login** (`proses_login`) — menentukan halaman mana
   (`pembeli_box` atau `penjual_box`) yang ditampilkan setelah login
   berhasil, berdasarkan peran yang dikembalikan oleh `cek_login`.
5. **Peluncuran aplikasi** — `app.launch()` menjalankan server Gradio
   sehingga UI dapat diakses melalui browser (atau tautan publik apabila
   dijalankan dengan `share=True`).

> **Catatan mengenai struktur sel:** seluruh definisi UI di atas berada
> dalam satu blok `with gr.Blocks() as app:` yang sama. Secara teknis,
> badan (*body*) dari sebuah blok `with` di Python tidak dapat dipisah ke
> beberapa sel notebook yang berbeda, sehingga—berbeda dengan sel-sel
> fungsi backend sebelumnya—sel UI ini tetap disatukan menjadi satu sel.
> Untuk menjaga kerapian, sel ini telah diberi indentasi yang konsisten
> dan komentar penanda antar-bagian (Login, Pembeli, Penjual) sehingga
> setiap bagian tetap mudah ditelusuri.

**Implikasi/interpretasi output:** menjalankan sel ini akan mencetak URL
lokal (dan URL publik apabila `share=True`) tempat aplikasi Gradio dapat
diakses. Selama sel ini berjalan (server aktif), sel-sel berikutnya pada
notebook tidak akan dieksekusi hingga server dihentikan secara manual
(misalnya dengan menghentikan eksekusi sel atau memanggil `app.close()`).


In [ ]:
# === GRADIO APP ===
with gr.Blocks() as app:
    gr.Markdown("## Kantin Terintegrasi - Login")

    # ---------------------------------------------------------------
    # BAGIAN 1: LOGIN & REGISTRASI
    # ---------------------------------------------------------------
    with gr.Row():
        email_in = gr.Textbox(label="Email")
        password_in = gr.Textbox(label="Password", type="password")
        login_btn = gr.Button("Login")

    status_out = gr.Textbox(label="Status", interactive=False)

    with gr.Accordion("Belum punya akun? Daftar di sini", open=False):
        role_reg = gr.Dropdown(choices=["pembeli", "penjual"], label="Daftar sebagai")
        email_reg = gr.Textbox(label="Email")
        password_reg = gr.Textbox(label="Password", type="password")
        nama_reg = gr.Textbox(label="Nama")
        npm_reg = gr.Textbox(label="NPM", visible=False)
        field1_reg = gr.Textbox(label="", visible=False)  # Jurusan atau Nama Kios
        field2_reg = gr.Textbox(label="", visible=False)  # No HP atau Nama Pedagang
        reg_btn = gr.Button("Daftar")
        reg_status = gr.Textbox(label="Status Registrasi")

        def update_form_fields(role):
            """Menyesuaikan field form registrasi berdasarkan peran yang dipilih."""
            if role == "pembeli":
                return gr.update(visible=True), gr.update(label="Jurusan", visible=True), gr.update(label="No HP", visible=True)
            elif role == "penjual":
                return gr.update(visible=False), gr.update(label="Nama Kios", visible=True), gr.update(label="Nama Pedagang", visible=True)
            return gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)

        role_reg.change(fn=update_form_fields, inputs=role_reg, outputs=[npm_reg, field1_reg, field2_reg])

        reg_btn.click(register_akun,
                      inputs=[email_reg, password_reg, role_reg, nama_reg, field1_reg, field2_reg, npm_reg],
                      outputs=reg_status)

    hidden_id = gr.Textbox(visible=False, value="0")

    # ---------------------------------------------------------------
    # BAGIAN 2: HALAMAN PEMBELI
    # ---------------------------------------------------------------
    pembeli_box = gr.Column(visible=False)
    with pembeli_box:
        gr.Markdown("### Halaman Pembeli")

        pembeli_main_menu = gr.Column(visible=True)
        with pembeli_main_menu:
            btn_buka_profil = gr.Button("Lihat Profil")
            btn_buka_order = gr.Button("Order")

        # --- Sub-halaman: Profil ---
        profil_box = gr.Column(visible=False)
        with profil_box:
            btn_back_profil = gr.Button("\u2190 Back")
            out_profil = gr.Textbox(label="Profil", lines=4, interactive=False)
            btn_lihat_profil = gr.Button("Refresh Profil")
            email_baru = gr.Textbox(label="Email Baru")
            password_baru = gr.Textbox(label="Password Baru", type="password")
            telp_baru = gr.Textbox(label="Nomor Telepon Baru")
            btn_update_profil = gr.Button("Update Profil")
            out_update = gr.Textbox(label="Status Update", lines=2, interactive=False)

        # --- Sub-halaman: Order ---
        order_box = gr.Column(visible=False)
        with order_box:
            btn_back_order = gr.Button("\u2190 Back")
            dropdown_kios = gr.Dropdown(label="Pilih Kios", choices=list(get_list_kios().keys()))
            dropdown_menu = gr.Dropdown(label="Pilih Menu", choices=[])

            btn_refresh_kios = gr.Button("Refresh Kios")

            def refresh_kios_dropdown():
                kios_dict = get_list_kios()
                return gr.update(choices=list(kios_dict.keys()), value=None), gr.update(choices=[], value=None)

            btn_refresh_kios.click(fn=refresh_kios_dropdown, outputs=[dropdown_kios, dropdown_menu])

            def update_menu_dropdown(nama_kios):
                kios_dict = get_list_kios()
                id_kios = kios_dict.get(nama_kios)
                if not id_kios:
                    return gr.update(choices=[], value=None)
                menu_dict = get_menu_by_kios(id_kios)
                return gr.update(choices=list(menu_dict.keys()), value=None)

            dropdown_kios.change(fn=update_menu_dropdown, inputs=dropdown_kios, outputs=dropdown_menu)

            out_cart = gr.Textbox(label="Isi Cart", lines=6)

            btn_tambah_cart = gr.Button("Tambah ke Cart")
            out_cart_msg = gr.Textbox(label="Status Cart", lines=2)

            def tambah_ke_cart_dan_lihat(nama_menu):
                msg = tambah_ke_cart(nama_menu)
                return msg, lihat_cart()

            btn_tambah_cart.click(fn=tambah_ke_cart_dan_lihat, inputs=dropdown_menu, outputs=[out_cart_msg, out_cart])

            btn_kosongkan = gr.Button("Kosongkan Cart")

            def kosongkan_cart_dan_lihat():
                msg = kosongkan_cart()
                return msg, lihat_cart()

            btn_kosongkan.click(fn=kosongkan_cart_dan_lihat, outputs=[out_cart_msg, out_cart])

            dropdown_metode = gr.Dropdown(label="Pilih Metode Pembayaran", choices=["Tunai", "Non Tunai"])
            out_metode = gr.Textbox(label="Metode Dipilih", lines=1)
            dropdown_metode.change(fn=simpan_metode, inputs=dropdown_metode, outputs=out_metode)

            btn_konfirmasi = gr.Button("Confirm Order")
            out_confirm = gr.Textbox(label="Status Order")
            btn_konfirmasi.click(fn=konfirmasi_order, inputs=hidden_id, outputs=out_confirm)

        # --- Navigasi antar sub-halaman pembeli ---
        def buka_profil():
            return gr.update(visible=False), gr.update(visible=True)

        def tutup_profil():
            return gr.update(visible=True), gr.update(visible=False)

        def buka_order():
            return gr.update(visible=False), gr.update(visible=True)

        def tutup_order():
            return gr.update(visible=True), gr.update(visible=False)

        btn_buka_profil.click(fn=buka_profil, outputs=[pembeli_main_menu, profil_box])
        btn_back_profil.click(fn=tutup_profil, outputs=[pembeli_main_menu, profil_box])
        btn_lihat_profil.click(fn=ambil_profil_pembeli, inputs=hidden_id, outputs=out_profil)
        btn_update_profil.click(fn=ubah_profil_pembeli, inputs=[hidden_id, email_baru, password_baru, telp_baru], outputs=out_update)

        btn_buka_order.click(fn=buka_order, outputs=[pembeli_main_menu, order_box])
        btn_back_order.click(fn=tutup_order, outputs=[pembeli_main_menu, order_box])

    # ---------------------------------------------------------------
    # BAGIAN 3: HALAMAN PENJUAL
    # ---------------------------------------------------------------
    penjual_box = gr.Column(visible=False)
    with penjual_box:
        gr.Markdown("### Halaman Penjual")

        # --- Tab: Lihat Data ---
        with gr.Tab("Lihat Data"):
            btn_menu_all = gr.Button("Lihat Menu Saya")
            btn_kios_all = gr.Button("Lihat Semua Kios")
            btn_trx_all = gr.Button("Lihat Transaksi Saya")
            out_lihat = gr.Textbox(label="Hasil", lines=15)

            btn_menu_all.click(fn=lihat_semua_menu_penjual, inputs=hidden_id, outputs=out_lihat)
            btn_kios_all.click(fn=lihat_semua_kios, outputs=out_lihat)
            btn_trx_all.click(fn=lihat_trx_penjual, inputs=hidden_id, outputs=out_lihat)

        # --- Tab: Tambah Menu ---
        with gr.Tab("Tambah Menu"):
            nama_add_menu = gr.Textbox(label="Nama Menu")
            kategori_add_menu = gr.Dropdown(label="Kategori", choices=ALLOWED_CATEGORIES)
            harga_add_menu = gr.Textbox(label="Harga")
            btn_add_menu = gr.Button("Tambah Menu")
            out_add_menu = gr.Textbox(label="Status")

            def tambah_menu(nama, kategori, harga, id_akun):
                try:
                    if kategori not in ALLOWED_CATEGORIES:
                        return "Kategori tidak valid. Pilih dari daftar yang tersedia."

                    id_kios = get_id_kios_from_akun(int(id_akun))
                    if not id_kios:
                        return "ID Kios tidak ditemukan."

                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()

                    # Skema penomoran ID Menu diturunkan dari ID Kios,
                    # sehingga setiap kios memiliki rentang ID Menu tersendiri.
                    prefix = (id_kios % 100) * 1000
                    cursor.execute('SELECT MAX("ID Menu") FROM Menu WHERE "ID Menu" BETWEEN ? AND ?', (prefix, prefix + 999))
                    max_id = cursor.fetchone()[0]
                    next_id = prefix + 1 if max_id is None else max_id + 1

                    cursor.execute('INSERT INTO Menu ("ID Menu", "Menu", Kategori, Harga, "ID Kios") VALUES (?, ?, ?, ?, ?)',
                                   (next_id, nama, kategori, int(harga), id_kios))
                    conn.commit()
                    conn.close()
                    return f"Menu berhasil ditambahkan dengan ID {next_id}."
                except ValueError:
                    return "Harga harus berupa angka."
                except Exception as e:
                    return f"Error: {e}"

            btn_add_menu.click(tambah_menu, inputs=[nama_add_menu, kategori_add_menu, harga_add_menu, hidden_id], outputs=out_add_menu)

        # --- Tab: Profil Kios ---
        with gr.Tab("Profil Kios"):
            btn_lihat_profil_kios = gr.Button("Lihat Profil Saya")
            out_profil_kios = gr.Textbox(label="Profil Kios", lines=5)
            btn_lihat_profil_kios.click(fn=lihat_profil_kios, inputs=[hidden_id], outputs=out_profil_kios)

        # --- Tab: Edit Menu ---
        with gr.Tab("Edit Menu"):
            id_menu_edit = gr.Textbox(label="ID Menu")
            nama_edit = gr.Textbox(label="Nama Menu Baru")
            kategori_edit = gr.Dropdown(label="Kategori Baru", choices=ALLOWED_CATEGORIES)
            harga_edit = gr.Textbox(label="Harga Baru")
            btn_update_menu = gr.Button("Update Menu")
            out_update_menu = gr.Textbox(label="Status")

            def update_menu(id_menu, nama, kategori, harga, id_akun):
                try:
                    if kategori not in ALLOWED_CATEGORIES:
                        return "Kategori tidak valid. Pilih dari daftar yang tersedia."

                    id_kios = get_id_kios_from_akun(int(id_akun))
                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()
                    cursor.execute('SELECT "ID Kios" FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    menu_kios = cursor.fetchone()
                    if not menu_kios or menu_kios[0] != id_kios:
                        conn.close()
                        return "Anda tidak memiliki akses untuk mengedit menu ini."
                    cursor.execute('UPDATE Menu SET "Menu"=?, Kategori=?, Harga=? WHERE "ID Menu"=?',
                                   (nama, kategori, int(harga), int(id_menu)))
                    conn.commit()
                    conn.close()
                    return "Menu berhasil diperbarui."
                except ValueError:
                    return "Harga harus berupa angka."
                except Exception as e:
                    return f"Error: {e}"

            btn_update_menu.click(update_menu, inputs=[id_menu_edit, nama_edit, kategori_edit, harga_edit, hidden_id], outputs=out_update_menu)

        # --- Tab: Hapus Menu ---
        with gr.Tab("Hapus Menu"):
            id_menu_hapus = gr.Textbox(label="ID Menu")
            btn_hapus_menu = gr.Button("Hapus Menu")
            out_hapus_menu = gr.Textbox(label="Status")

            def hapus_menu(id_menu, id_akun):
                try:
                    id_kios = get_id_kios_from_akun(int(id_akun))
                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()
                    cursor.execute('SELECT "ID Kios" FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    menu_kios = cursor.fetchone()
                    if not menu_kios or menu_kios[0] != id_kios:
                        conn.close()
                        return "Anda tidak memiliki akses untuk menghapus menu ini."
                    cursor.execute('DELETE FROM Menu WHERE "ID Menu"=?', (int(id_menu),))
                    conn.commit()
                    conn.close()
                    return "Menu berhasil dihapus."
                except Exception as e:
                    return f"Error: {e}"

            btn_hapus_menu.click(hapus_menu, inputs=[id_menu_hapus, hidden_id], outputs=out_hapus_menu)

        # --- Tab: Kelola Kios ---
        with gr.Tab("Kelola Kios"):
            nama_kios_kelola = gr.Textbox(label="Nama Kios Baru")
            nama_pedagang_kelola = gr.Textbox(label="Nama Pedagang Baru")
            email_baru_akun = gr.Textbox(label="Email Akun Baru")
            password_baru_akun = gr.Textbox(label="Password Akun Baru", type="password")
            btn_update_kios = gr.Button("Update Kios dan Akun")
            out_kios = gr.Textbox(label="Status")

            def update_kios_dan_akun(nama_kios, nama_pedagang, email_baru, password_baru, id_akun):
                try:
                    # Validasi input tidak kosong
                    for field_name, value in {
                        "Nama Kios": nama_kios,
                        "Nama Pedagang": nama_pedagang,
                        "Email Baru": email_baru,
                        "Password Baru": password_baru,
                        "ID Akun": id_akun
                    }.items():
                        if not str(value).strip():
                            return f"Field '{field_name}' tidak boleh kosong."

                    id_akun = int(id_akun)
                    id_kios = get_id_kios_from_akun(id_akun)
                    if not id_kios:
                        return "ID Kios tidak ditemukan."

                    conn = sqlite3.connect(DB_PATH)
                    cursor = conn.cursor()

                    # Cek apakah email baru sudah dipakai oleh akun lain
                    cursor.execute(
                        'SELECT 1 FROM Akun WHERE "Email"=? AND "ID Akun"!=?',
                        (email_baru.strip(), id_akun)
                    )
                    if cursor.fetchone():
                        conn.close()
                        return "Email sudah digunakan oleh akun lain."

                    # Cek apakah nama kios sudah dipakai oleh kios lain
                    cursor.execute(
                        'SELECT 1 FROM Kios WHERE "Nama Kios"=? AND "ID Kios"!=?',
                        (nama_kios.strip(), id_kios)
                    )
                    if cursor.fetchone():
                        conn.close()
                        return "Nama kios sudah digunakan oleh kios lain."

                    # Lanjutkan update karena valid
                    cursor.execute(
                        'UPDATE Kios SET "Nama Kios"=?, "Nama Pedagang"=? WHERE "ID Kios"=?',
                        (nama_kios.strip(), nama_pedagang.strip(), id_kios)
                    )

                    cursor.execute(
                        'UPDATE Akun SET "Email"=?, "Password"=? WHERE "ID Akun"=?',
                        (email_baru.strip(), password_baru.strip(), id_akun)
                    )

                    conn.commit()
                    conn.close()
                    return "Kios dan akun berhasil diperbarui."

                except Exception as e:
                    return f"Error: {e}"

            btn_update_kios.click(fn=update_kios_dan_akun,
                                  inputs=[nama_kios_kelola, nama_pedagang_kelola, email_baru_akun, password_baru_akun, hidden_id],
                                  outputs=out_kios)

    # ---------------------------------------------------------------
    # BAGIAN 4: LOGIKA NAVIGASI SETELAH LOGIN
    # ---------------------------------------------------------------
    def proses_login(email, password):
        id_akun, role = cek_login(email, password)
        if not id_akun:
            return "Email atau password salah.", gr.update(visible=False), gr.update(visible=False), ""
        if role == "penjual":
            return f"Login berhasil sebagai PENJUAL (ID: {id_akun})", gr.update(visible=False), gr.update(visible=True), str(id_akun)
        elif role == "pembeli":
            return f"Login berhasil sebagai PEMBELI (ID: {id_akun})", gr.update(visible=True), gr.update(visible=False), str(id_akun)
        else:
            return "Akun tidak dikenali.", gr.update(visible=False), gr.update(visible=False), ""

    login_btn.click(fn=proses_login,
                    inputs=[email_in, password_in],
                    outputs=[status_out, pembeli_box, penjual_box, hidden_id])


### 6.1 Menjalankan Aplikasi

Sel terakhir menjalankan `app.launch()` untuk memulai server Gradio.

**Implikasi/interpretasi output:** setelah dijalankan, Gradio akan
mencetak URL lokal (misalnya `http://127.0.0.1:7860`) yang dapat dibuka
melalui browser untuk mengakses aplikasi Kantin Terintegrasi secara
interaktif. Sel ini akan tetap berjalan (server aktif) sampai dihentikan
secara manual, sehingga sel-sel berikutnya (jika ada) baru akan
tereksekusi setelah server dihentikan.


In [ ]:
app.launch()
